In [ ]:
# Création de graphique avec Bokeh


In [1]:
import numpy as np
# Importation des modules nécessaires
from bokeh.plotting import figure, output_file, show

# Préparation des données
x = [1, 2, 3, 4, 5]
y = [6, 7, 2, 4, 5]

#Sorties des données en fichier HTML
output_file("lines.html")

# Création de l'objet figure
p = figure(title = "Simple line example", x_axis_label = 'X-axis', y_axis_label = 'Y-axis')

# Ajout de glyphes
p.line(x, y, legend_label="Temp.", line_width=2)

# Affichage du graphique
show(p)

## Créer des graphiques interactifs

Pour augmenter l’interactivité des graphiques, Bokeh permet l’intégration de widgets tels que des sliders, des boutons et des menus déroulants.

In [3]:
# Importation des modules nécessaires
from bokeh.plotting import figure, output_file, show
from bokeh.models import ColumnDataSource, Slider
from bokeh.layouts import column

# Définition de l'endroit où le graphique sera sauvegardé
output_file("simple_graph.html")

# Préparation des données
# Utilisation de ColumnDataSource pour une intégration facile avec des glyphes
data = ColumnDataSource(data={'x': [1, 2, 3, 4, 5], 'y': [3, 7, 2, 6, 4]})

# Création de la figure
# Initialise une figure avec un titre et des étiquettes d'axes
p = figure(title="Graphique simple", x_axis_label='X', y_axis_label='Y')

# Ajout de glyphes
# Ajoute une ligne reliant les points x et y
p.line(x='x', y='y', source=data, line_width=2)

# Création d'une disposition simple
# Organise le graphique dans une disposition de colonne simple
layout = column(p)

# Affichage du graphique
show(layout)

Fonctionne qu'avec le serveur de bokeh : `bokeh serve --show myapp.py`

---

### 1. Pourquoi votre deuxième graphique ne "fonctionne" pas ?

Il y a deux raisons principales, l'une technique et l'autre liée à la manière d'exécuter Bokeh :

1.  **Le mode d'exécution (Le serveur Bokeh) :** Votre deuxième code utilise `curdoc()` et `on_change`. Ces fonctions nécessitent un **serveur Bokeh** pour fonctionner. Si vous lancez ce script comme un fichier Python classique ou dans une cellule Jupyter standard, l'interactivité (le slider) ne répondra pas car il n'y a pas de "moteur" Python derrière pour calculer les nouvelles valeurs de `y` en temps réel.
2.  **Erreurs de syntaxe/logique dans le code :** * `x_range` attend une plage (ex: `[0, 10]`), pas une seule valeur.
    * Le module `Slider` n'était pas importé dans le deuxième bloc.
    * `np.linspace` nécessite un nombre de points (ex: 500) pour une courbe lisse.

---

### 2. Le code corrigé et fonctionnel

Pour tester ce code et voir le slider bouger la courbe, vous devez l'enregistrer dans un fichier nommé `wave.py` et le lancer dans votre terminal avec la commande suivante :
`bokeh serve --show wave.py`

Voici le code corrigé :

```python
import numpy as np
from bokeh.io import curdoc
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, Slider # Import Slider ajouté
from bokeh.layouts import column

# 1. Préparation des données plus précises
x = np.linspace(0, 10, 500) # De 0 à 10 avec 500 points pour la fluidité
y = np.sin(x)
source = ColumnDataSource(data={'x': x, 'y': y})

# 2. Création de la figure
# Correction de x_range pour voir toute la courbe
plot = figure(title="Exemple de vague dynamique",
              x_range=[0, 10],
              y_range=[-1.5, 1.5])

plot.line(x='x', y='y', source=source, line_width=3, color="navy")

# 3. Création du slider
slider = Slider(start=0.1, end=10, value=1, step=0.1, title="Fréquence")

# 4. Fonction de mise à jour (Callback)
def update_data(attr, old, new):
    f = slider.value # On récupère la valeur actuelle du curseur
    new_y = np.sin(f * x) # On recalcule les coordonnées Y
    source.data = {'x': x, 'y': new_y} # On remplace les données de la source

# 5. Liaison : dès que la 'value' du slider change, on appelle update_data
slider.on_change('value', update_data)

# 6. Mise en page et ajout au document
layout = column(slider, plot)
curdoc().add_root(layout)
curdoc().title = "Onde Interactive"
```

---

### 3. Explications des concepts clés

Pour mieux comprendre Bokeh, imaginez-le comme un système en trois couches :

#### A. Le `ColumnDataSource` (La Source de Données)
C'est le concept le plus important. Au lieu de passer des listes Python directement au graphique, on les met dans ce "panier".
* **Pourquoi ?** Parce que si vous changez les données à l'intérieur du panier (comme dans la fonction `update_data`), le graphique se met à jour automatiquement sans avoir à le redessiner entièrement.

#### B. Les Glyphes (`p.line`, `p.circle`)
Ce sont les formes visuelles. Vous liez un glyphe à une source. Dans votre code : `p.line(x='x', y='y', source=source)`. Cela dit à Bokeh : "Dessine une ligne en utilisant les colonnes 'x' et 'y' qui se trouvent dans `source`".

#### C. Le Serveur Bokeh vs Export statique
* **Graphique 1 (Statique) :** Vous utilisez `output_file` et `show`. Bokeh génère un fichier HTML "mort". Une fois créé, Python ne s'en occupe plus. C'est parfait pour des graphiques simples que l'on envoie par mail.
* **Graphique 2 (Dynamique) :** Vous utilisez `curdoc` (Current Document). Ici, Bokeh crée un lien vivant entre votre navigateur web et Python. Quand vous bougez le slider, le navigateur envoie un message à Python : *"Hé, la valeur a changé !"*. Python exécute `update_data` et renvoie les nouvelles coordonnées au navigateur.

